# Homework 08: GPT implementation

Refer to the corresponding lab and Rashcka's book (Ch. 4 and 5)

**Exercise 1 — Parameter counting**

Without running code, estimate the number of parameters in:
- One `FeedForward` block (hint: two linear layers)
- One `MultiHeadAttention` block (hint: four linear layers — Q, K, V, out_proj)

Then verify with code:

In [1]:
# ============================================================================
# Setup: imports + the GPT building blocks from the lab (08_GPT_Imp_Train)
# We re-define them here so this homework notebook is self-contained.
# ============================================================================
import torch
import torch.nn as nn
import tiktoken

torch.manual_seed(123)

# Base GPT-2 124M configuration (same as the lab)
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # BPE vocabulary size (tiktoken gpt2)
    "context_length": 1024,  # Maximum sequence length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of transformer blocks
    "drop_rate": 0.1,        # Dropout probability
    "qkv_bias": False,       # No bias in Q/K/V projections
}


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    """Position-wise feed-forward: expand to 4x, non-linearity, contract back."""
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # Linear 1
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),  # Linear 2
        )

    def forward(self, x):
        return self.layers(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2).contiguous()
        context_vec = context_vec.view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"], num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.drop_shortcut(self.att(self.norm1(x))) + shortcut  # attention + residual
        shortcut = x
        x = self.drop_shortcut(self.ff(self.norm2(x))) + shortcut    # feed-forward + residual
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


tokenizer = tiktoken.get_encoding("gpt2")

# ============================================================================
# Exercise 1 — Parameter counting (manual estimate first, then verify)
# ============================================================================
emb = GPT_CONFIG_124M["emb_dim"]   # 768

# ---- Manual estimate: FeedForward ----
# Two Linear layers (each Linear has weights = in*out, plus a bias of size out):
#   Linear 1:  768 -> 3072  =>  768*3072 weights + 3072 biases
#   Linear 2: 3072 ->  768  =>  3072*768 weights +  768 biases
ff_estimate = (emb * 4 * emb + 4 * emb) + (4 * emb * emb + emb)
print(f"FeedForward — manual estimate: {ff_estimate:,}")  # expect 4,722,432

# ---- Manual estimate: MultiHeadAttention ----
# Four Linear layers. Q, K, V have NO bias (qkv_bias=False); out_proj HAS a bias.
#   W_query: 768x768, W_key: 768x768, W_value: 768x768   (no bias)
#   out_proj: 768x768 weights + 768 biases
mha_estimate = 3 * (emb * emb) + (emb * emb + emb)
print(f"MultiHeadAttention — manual estimate: {mha_estimate:,}")  # expect 2,360,064

# ---- Verify with code ----
ff = FeedForward(GPT_CONFIG_124M)
mha = MultiHeadAttention(d_in=emb, d_out=emb,
                         context_length=GPT_CONFIG_124M["context_length"],
                         dropout=0.0, num_heads=GPT_CONFIG_124M["n_heads"],
                         qkv_bias=False)

ff_actual = sum(p.numel() for p in ff.parameters())
# Count only learnable params for MHA (exclude the non-learnable causal mask buffer)
mha_actual = sum(p.numel() for p in mha.parameters())

print(f"\nFeedForward — actual:        {ff_actual:,}   match: {ff_actual == ff_estimate}")
print(f"MultiHeadAttention — actual: {mha_actual:,}   match: {mha_actual == mha_estimate}")

FeedForward — manual estimate: 4,722,432
MultiHeadAttention — manual estimate: 2,360,064

FeedForward — actual:        4,722,432   match: True
MultiHeadAttention — actual: 2,360,064   match: True


**Exercise 2 — Scaling up**

GPT-2 comes in four sizes. Modify the config to implement each one and count its parameters:

| Model | emb_dim | n_layers | n_heads |
|-------|---------|----------|---------|
| Small | 768 | 12 | 12 |
| Medium | 1024 | 24 | 16 |
| Large | 1280 | 36 | 20 |
| XL | 1600 | 48 | 25 |

In [2]:
# ============================================================================
# Exercise 2 — Scaling up to the four GPT-2 sizes
# ============================================================================
# Each size only changes emb_dim, n_layers and n_heads. Everything else
# (vocab_size, context_length, dropout, bias) stays the same.
gpt2_sizes = {
    "Small  (124M)": {"emb_dim": 768,  "n_layers": 12, "n_heads": 12},
    "Medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "Large  (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "XL    (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


def make_config(overrides):
    """Start from the base 124M config and apply the size-specific overrides."""
    cfg = GPT_CONFIG_124M.copy()
    cfg.update(overrides)
    return cfg


def count_params(model):
    """Total params, plus the 'weight-tied' count used by the original GPT-2
    (output head reuses the token-embedding matrix, so we subtract it once)."""
    total = sum(p.numel() for p in model.parameters())
    out_head = sum(p.numel() for p in model.out_head.parameters())
    return total, total - out_head


print(f"{'Model':<14}{'Total params':>16}{'Weight-tied':>16}{'Size (MB, fp32)':>18}")
print("-" * 64)
for name, overrides in gpt2_sizes.items():
    model = GPTModel(make_config(overrides))
    total, tied = count_params(model)
    size_mb = total * 4 / (1024 ** 2)          # 4 bytes per float32 parameter
    print(f"{name:<14}{total:>16,}{tied:>16,}{size_mb:>16.1f} MB")
    del model                                   # free memory before building the next size

Model             Total params     Weight-tied   Size (MB, fp32)
----------------------------------------------------------------
Small  (124M)      163,009,536     124,412,160           621.8 MB
Medium (355M)      406,212,608     354,749,440          1549.6 MB
Large  (774M)      838,220,800     773,891,840          3197.6 MB
XL    (1558M)    1,637,792,000   1,557,380,800          6247.7 MB


**Exercise 3 — Temperature sampling**

The `generate_text_simple` function always picks the *most likely* token (greedy decoding). This produces deterministic but often repetitive text.

A common alternative is **temperature sampling**: divide logits by a temperature τ before applying softmax. Lower τ → more deterministic; higher τ → more random.

Modify `generate_text_simple` to accept a `temperature` argument and sample from the distribution instead of always taking `argmax`. Test with temperatures 0.5, 1.0, and 2.0 on the same prompt.

Hint: You can check Rashcka's book to get an intuition

In [3]:
# ============================================================================
# Exercise 3 — Temperature sampling
# ============================================================================
def generate_text(model, idx, max_new_tokens, context_size, temperature=0.0):
    """Autoregressive generation with optional temperature sampling.

    temperature == 0  -> greedy decoding (argmax), fully deterministic.
    temperature  > 0  -> divide logits by tau, softmax, then SAMPLE.
                         tau < 1 sharpens the distribution (more confident/repetitive),
                         tau > 1 flattens it (more diverse/random).
    """
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]            # crop to context window
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]                    # last position -> next-token logits

        if temperature > 0.0:
            # Scale logits by temperature, turn into a probability distribution,
            # then draw one token at random according to those probabilities.
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)   # (batch, 1)
        else:
            # Greedy: always take the single most likely token.
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)      # append and continue
    return idx


# --- Helpers to move between text and token-id tensors ---
def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0)   # add batch dim

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())     # drop batch dim


# --- Test the same prompt at three temperatures on an (untrained) model ---
# NOTE: this model has random weights, so the text will be gibberish — the point
# here is only to observe HOW temperature changes the sampling behaviour.
torch.manual_seed(123)
demo_model = GPTModel(GPT_CONFIG_124M)
demo_model.eval()

prompt = "The Supreme Court held that"
start = text_to_token_ids(prompt, tokenizer)

for tau in [0.5, 1.0, 2.0]:
    torch.manual_seed(123)   # same seed each time so differences come only from tau
    out = generate_text(demo_model, start, max_new_tokens=15,
                        context_size=GPT_CONFIG_124M["context_length"],
                        temperature=tau)
    print(f"--- temperature = {tau} ---")
    print(token_ids_to_text(out, tokenizer).replace("\n", " "))
    print()

--- temperature = 0.5 ---
The Supreme Court held thatden rendering purseTempfinalTweet Nokia Southampton sanctions installation reductorsitionernel planes

--- temperature = 1.0 ---
The Supreme Court held thatenegger rendering purseTempfinalTweet Nokiabanks sanctions installation reduFsitionernel planes

--- temperature = 2.0 ---
The Supreme Court held thatenegger rendering GroundTempfinalTweetntonbanks sanctions installation reduFsitionernel planes



**Exercise 4 - Training GPT**

Use the code from the lab to train your own GPT-2 model, this time use another dataset (if possible, one bigger or more interesting).

Deploy the model and ask it to predict the next word for a relevant sentence you choose

In [4]:
# ============================================================================
# Exercise 4 — Train your own GPT-2 on a new (bigger) dataset
# Dataset: "Tiny Shakespeare" (~1.1 MB, ~338k tokens) — much larger and more
# varied than "The Verdict" used in the lab.
# ============================================================================
import urllib.request
from torch.utils.data import Dataset, DataLoader

# ---- 1. Download the dataset --------------------------------------------------
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "tiny-shakespeare.txt")
with open("tiny-shakespeare.txt", "r", encoding="utf-8") as f:
    text_data = f.read()
print(f"Characters: {len(text_data):,}  |  Tokens: {len(tokenizer.encode(text_data)):,}")


# ---- 2. Dataset + DataLoader (sliding window of token chunks) -----------------
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt)
        # Slide a window over the text; target is the input shifted right by one.
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)


# ---- 3. Smaller config so training is feasible on CPU -------------------------
GPT_CONFIG_TRAIN = {
    "vocab_size": 50257,
    "context_length": 256,   # shorter context = faster training
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}
ctx = GPT_CONFIG_TRAIN["context_length"]

# 90 / 10 train/validation split
split_idx = int(0.90 * len(text_data))
torch.manual_seed(123)
train_loader = create_dataloader_v1(text_data[:split_idx], batch_size=4,
                                    max_length=ctx, stride=ctx, shuffle=True)
val_loader = create_dataloader_v1(text_data[split_idx:], batch_size=4,
                                  max_length=ctx, stride=ctx,
                                  shuffle=False, drop_last=False)
print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))


# ---- 4. Loss + training utilities (from the lab) ------------------------------
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (xb, yb) in enumerate(data_loader):
        if i >= num_batches:
            break
        total_loss += calc_loss_batch(xb, yb, model, device).item()
    return total_loss / num_batches


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, val_loss


def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context):
    train_losses, val_losses, track_tokens = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(xb, yb, model, device)
            loss.backward()                 # backprop
            optimizer.step()                # update weights
            tokens_seen += xb.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                tr, va = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(tr); val_losses.append(va); track_tokens.append(tokens_seen)
                print(f"Ep {epoch+1} (step {global_step:05d}): train {tr:.3f}, val {va:.3f}")
        # Sample a continuation after each epoch to watch the model improve
        model.eval()
        ids = generate_text(model, text_to_token_ids(start_context, tokenizer).to(device),
                            max_new_tokens=30, context_size=ctx, temperature=0.0)
        print("  sample:", token_ids_to_text(ids, tokenizer).replace("\n", " "))
        model.train()
    return train_losses, val_losses, track_tokens


# ---- 5. Train ----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_TRAIN).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)

# A few epochs is enough to clearly bring the loss down; raise num_epochs for
# better text (training is the slow part — minutes per epoch on CPU).
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=10, eval_freq=50, eval_iter=5,
    start_context="To be, or not to be",
)


# ---- 6. Deploy: predict the next word for a sentence I choose -----------------
def predict_next_word(model, prompt, temperature=0.0):
    """Generate the single most likely next token for `prompt`."""
    model.eval()
    ids = generate_text(model, text_to_token_ids(prompt, tokenizer).to(device),
                        max_new_tokens=1, context_size=ctx, temperature=temperature)
    full = token_ids_to_text(ids, tokenizer)
    next_word = full[len(prompt):]              # the part the model appended
    print(f"Prompt: {prompt!r}")
    print(f"Predicted next token: {next_word!r}")
    return next_word

predict_next_word(model, "My lord, the king is")

Characters: 1,115,394  |  Tokens: 338,025
Train batches: 294 | Val batches: 35
Ep 1 (step 00000): train 9.364, val 9.314


KeyboardInterrupt: 